### FEATURE ENGINEERING

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
df = pd.read_csv("C:\\Users\\VP678WV\\OneDrive - EY\\Documents\\Delivery_Delay\\data\\raw\\mim2025_delay_sample.csv")
df.head()

,payment_type,profit_per_order,sales_per_customer,category_id,category_name,customer_city,customer_country,customer_id,customer_segment,customer_state,...,order_region,order_state,order_status,product_card_id,product_category_id,product_name,product_price,shipping_date,shipping_mode,label
0,PAYMENT,101.010895,195.02570,46,Indoor/Outdoor Games,Milford,EE. UU.,3736.8790,Home Office,CT,...,Western Europe,Vienna,PENDING_PAYMENT,1014,46,O'Brien Men's Neoprene Life Vest,49.98,42322.746,Standard Class,0
1,TRANSFER,85.423610,245.20793,46,Indoor/Outdoor Games,Caguas,Puerto Rico,4268.8580,Consumer,PR,...,South America,Buenos Aires,PENDING,1014,46,O'Brien Men's Neoprene Life Vest,49.98,42340.965,Standard Class,1
2,PAYMENT,261.173770,456.55527,9,Cardio Equipment,Caguas,Puerto Rico,7772.3945,Consumer,PR,...,Western Europe,West Flanders,PENDING_PAYMENT,191,9,Nike Men's Free 5.0+ Running Shoe,99.99,42061.550,Standard Class,-1
3,DEBIT,-52.374670,191.65901,48,Water Sports,Caguas,Puerto Rico,9977.6810,Consumer,PR,...,South America,O'Higgins,COMPLETE,1073,48,Pelican Sunstream 100 Kayak,199.99,42883.145,Standard Class,-1
4,DEBIT,55.085342,187.45561,48,Water Sports,Caguas,Puerto Rico,8920.0180,Consumer,PR,...,East of USA,New York,ON_HOLD,1073,48,Pelican Sunstream 100 Kayak,199.99,42950.610,Standard Class,0


1. Remove or Not Useful Features
These features are IDs or have high cardinality and no direct semantic meaning for modeling (especially with tree-based models like Random Forest):

| Feature                  | Reason                                            |
| ------------------------ | ------------------------------------------------- |
| `customer_id`            | Arbitrary ID, no pattern — drop                   |
| `order_id`               | Arbitrary ID, unique per order — drop             |
| `order_item_id`          | Arbitrary ID — drop                               |
| `order_customer_id`      | Same as `customer_id`, redundant — drop           |
| `order_item_cardprod_id` | Arbitrary code — drop                             |
| `product_card_id`        | Arbitrary product code — drop                     |
| `product_category_id`    | Redundant with `category_id` — drop one of them   |
| `department_id`          | Likely numeric code — not semantically meaningful |
| `category_id`            | Redundant if `category_name` used — drop one      |
| `customer_zipcode`       | Textual zip, too sparse — drop                    |


2. Features That Can Be Used As-Is After Normalization
These are pure numerical features that can go directly into models like XGBoost or can be normalized for logistic regression, etc.

| Feature                    | Notes                                   |
| -------------------------- | --------------------------------------- |
| `profit_per_order`         | Keep                                    |
| `sales_per_customer`       | Keep                                    |
| `order_item_discount`      | Keep                                    |
| `order_item_discount_rate` | Keep                                    |
| `order_item_product_price` | Keep                                    |
| `order_item_profit_ratio`  | Keep                                    |
| `order_item_quantity`      | Keep                                    |
| `sales`                    | Keep                                    |
| `order_item_total_amount`  | Keep                                    |
| `order_profit_per_order`   | Keep                                    |
| `product_price`            | Keep                                    |
| `latitude`                 | Optional; keep if location is important |
| `longitude`                | Optional; keep if location is important |


3. Categorical Features – One-Hot or Label Encode
These are categorical with a reasonable number of unique values, suitable for encoding.

| Feature            | Encoding                                     | Notes |
| ------------------ | -------------------------------------------- | ----- |
| `payment_type`     | One-hot or Label                             |       |
| `customer_segment` | One-hot                                      |       |
| `customer_city`    | Label (high cardinality)                     |       |
| `customer_country` | One-hot                                      |       |
| `customer_state`   | Label                                        |       |
| `market`           | One-hot                                      |       |
| `order_city`       | Label                                        |       |
| `order_country`    | One-hot                                      |       |
| `order_state`      | Label                                        |       |
| `order_status`     | One-hot                                      |       |
| `order_region`     | One-hot                                      |       |
| `shipping_mode`    | One-hot                                      |       |
| `label`            | Already target class (categorical: -1, 0, 1) |       |


4. Textual Features – Can Be Embedded (Optional)
You can extract semantic embeddings from these fields if you use deep models (like neural nets or transformer-based classifiers). Otherwise, skip them for tree-based models like Random Forest.

| Feature            | Notes                                                         |
| ------------------ | ------------------------------------------------------------- |
| `category_name`    | Embed if categories are descriptive                           |
| `department_name`  | Embed or use as categorical if limited values                 |
| `product_name`     | Can be embedded (e.g., using sentence-transformers or TF-IDF) |
| `customer_zipcode` | Skip or embed if used with geographic context                 |


5. Datetime Columns — Extract Useful Features
Don't use raw datetime. Instead, extract features like:

| Column                                             | Extracted Features                          |
| -------------------------------------------------- | ------------------------------------------- |
| `order_date`                                       | Year, Month, Day, Weekday, Hour (if needed) |
| `shipping_date`                                    | Same as above                               |
| `shipping_delay_days = shipping_date - order_date` | Derived feature: actual delivery duration   |


In [4]:
# Count how many items are in each order
order_counts = df['order_id'].value_counts()

# Basic stats
print(order_counts.describe())

# Count how many orders have only one item
single_item_orders = (order_counts == 1).sum()
multi_item_orders = (order_counts > 1).sum()
total_orders = len(order_counts)

print(f"🟢 Total Orders: {total_orders}")
print(f"✅ Orders with 1 item: {single_item_orders} ({single_item_orders / total_orders:.2%})")
print(f"🔁 Orders with multiple items: {multi_item_orders} ({multi_item_orders / total_orders:.2%})")

count    15522.000000
mean         1.001675
std          0.040894
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max          2.000000
Name: count, dtype: float64
🟢 Total Orders: 15522
✅ Orders with 1 item: 15496 (99.83%)
🔁 Orders with multiple items: 26 (0.17%)


In [4]:
order_counts = df['order_id'].value_counts()
multi_item_orders = order_counts[order_counts > 1]

df_multi = df[df['order_id'].isin(multi_item_orders.index)]
df_multi.head(10)   # See first 10 rows

,payment_type,profit_per_order,sales_per_customer,category_id,category_name,customer_city,customer_country,customer_id,customer_segment,customer_state,...,order_region,order_state,order_status,product_card_id,product_category_id,product_name,product_price,shipping_date,shipping_mode,label
390,CASH,67.738060,177.41005,48,Water Sports,Caguas,Puerto Rico,4308.89700,Consumer,PR,...,Western Europe,North Rhine-Westphalia,CLOSED,1073,48,Pelican Sunstream 100 Kayak,199.99,42986.062,First Class,1
780,DEBIT,33.184300,129.99000,18,Men's Footwear,Baltimore,EE. UU.,6855.71630,Home Office,MD,...,Western Europe,Saxony,COMPLETE,403,18,Nike Men's CJ Elite 2 TD Football Cleat,129.99,42088.574,Standard Class,-1
800,TRANSFER,34.904220,371.00305,45,Fishing,Cicero,EE. UU.,2265.03270,Consumer,IL,...,US Center,Michigan,PROCESSING,1004,45,Field & Stream Sportsman 16 Gun Fire Safe,399.98,42510.960,Standard Class,1
882,PAYMENT,95.095140,227.53032,43,Camping & Hiking,Caguas,Puerto Rico,3123.96660,Corporate,PR,...,East of USA,New York,PAYMENT_REVIEW,957,43,Diamondback Women's Serene Classic Comfort Bi,299.98,42522.990,Second Class,1
982,CASH,92.878654,199.04110,48,Water Sports,Caguas,Puerto Rico,12164.10100,Home Office,PR,...,West of USA,Washington,CLOSED,1073,48,Pelican Sunstream 100 Kayak,199.99,42867.754,Second Class,1
1556,CASH,-90.280090,236.16000,17,Cleats,Caguas,Puerto Rico,8002.42300,Consumer,PR,...,South Asia,Maharashtra,CLOSED,365,17,Perfect Fitness Perfect Rip Deck,59.99,42763.133,Standard Class,1
2352,DEBIT,154.555280,363.96146,45,Fishing,Caguas,Puerto Rico,620.88873,Consumer,PR,...,Eastern Europe,Lublin,COMPLETE,1004,45,Field & Stream Sportsman 16 Gun Fire Safe,399.98,42465.258,Standard Class,1
2494,DEBIT,71.053910,173.99000,48,Water Sports,Lancaster,EE. UU.,4003.97580,Consumer,SC,...,West Africa,Kaduna,COMPLETE,1073,48,Pelican Sunstream 100 Kayak,199.99,42036.420,Standard Class,-1
2598,TRANSFER,20.549015,129.99000,46,Indoor/Outdoor Games,Caguas,Puerto Rico,5225.43950,Consumer,PR,...,Central America,Choluteca,PROCESSING,1014,46,O'Brien Men's Neoprene Life Vest,49.98,42872.510,Standard Class,1
3069,DEBIT,69.828930,269.97403,17,Cleats,Fort Worth,EE. UU.,5894.06740,Home Office,TX,...,West Asia,Adana,COMPLETE,365,17,Perfect Fitness Perfect Rip Deck,59.99,42603.730,Standard Class,1


In [5]:
import pandas as pd

# Assuming your DataFrame is called df
# Example: Check if 'product_price' and 'order_item_product_price' are duplicates
duplicate_check_1 = (df['product_price'] == df['order_item_product_price']).all()
print("Are 'product_price' and 'order_item_product_price' identical?", duplicate_check_1)

# Check if 'profit_per_order' and 'order_profit_per_order' are duplicates
duplicate_check_2 = (df['profit_per_order'] == df['order_profit_per_order']).all()
print("Are 'profit_per_order' and 'order_profit_per_order' identical?", duplicate_check_2)

Are 'product_price' and 'order_item_product_price' identical? True
Are 'profit_per_order' and 'order_profit_per_order' identical? False


In [6]:
# Display mismatched rows
mismatched = df[df['profit_per_order'] != df['order_profit_per_order']]
print(mismatched[['order_id', 'profit_per_order', 'order_profit_per_order']].head(10))

     order_id  profit_per_order  order_profit_per_order
0  22757.0400        101.010895               91.143528
1   7238.8037         85.423610               81.642330
2  14280.5270        261.173770              230.276970
3  60556.4340        -52.374670              -37.619053
4  45007.4200         55.085342               58.277086
5  60981.9600         33.196470               28.798080
6  12941.9350         80.954544               82.786200
7  70438.3360         42.726772               58.797060
8  76295.0000          8.021677                7.598100
9  42690.0620          6.733015                6.899200


In [7]:
# Check the distribution of differences
df['profit_diff'] = df['profit_per_order'] - df['order_profit_per_order']
print(df['profit_diff'].describe())

count    15548.000000
mean        -0.558500
std         64.364190
min      -3176.891700
25%         -3.669768
50%          0.223191
75%          4.041526
max        558.471240
Name: profit_diff, dtype: float64


In [8]:
!pip install geopy

In [ ]:
from geopy.geocoders import Nominatim
import pandas as pd
import time

geolocator = Nominatim(user_agent="geoapi")

# Create a DataFrame of unique delivery locations
unique_locs = df[['order_city', 'order_country']].drop_duplicates()

# Function to fetch coordinates
def get_coordinates(city, country):
    try:
        location = geolocator.geocode(f"{city}, {country}", timeout=10)
        if location:
            return pd.Series([location.latitude, location.longitude])
    except:
        return pd.Series([None, None])
    return pd.Series([None, None])

# Apply the function with a delay to avoid rate limiting
unique_locs[['dest_latitude', 'dest_longitude']] = unique_locs.apply(
    lambda row: get_coordinates(row['order_city'], row['order_country']), axis=1
)


In [26]:
missing_coords = unique_locs[unique_locs['dest_latitude'].isnull() | unique_locs['dest_longitude'].isnull()]
print(f"Number of locations with missing coordinates: {len(missing_coords)}")
# Show failed city-country combinations
print(missing_coords[['order_city', 'order_country']])

Number of locations with missing coordinates: 14
                             order_city       order_country
248                        Yamoussoukro        Marfil Coast
454                          Kryvyy Rih             Ukraine
903                         Wadi as Sir              Jordan
950    Fresnillo de Gonzalez Echeverria              Mexico
1649                            Abidjan        Marfil Coast
1695                             Gagnoa        Marfil Coast
1782                         Bene Beraq              Israel
2139                       Taldyqorghan     Kazajist√Ø¬ø¬Ωn
3610                          K'ut'aisi             Georgia
4574                            Cortama               Spain
4964                        Thies Nones             Senegal
5028                             Bouake        Marfil Coast
8903                            Brikama  Republic of Gambia
10015                             Daloa        Marfil Coast


In [34]:
# Fix common incorrect country names
country_fixes = {
    "Marfil Coast": "Ivory Coast",  # Official: Côte d'Ivoire
    "Kazajist√Ø¬ø¬Ωn": "Kazakhstan",
    "Republic of Gambia": "Gambia"
}

unique_locs['order_country'] = unique_locs['order_country'].replace(country_fixes)
df['order_country'] = df['order_country'].replace(country_fixes)

In [35]:
city_fixes = {
    "Fresnillo de Gonzalez Echeverria": "Fresnillo",
    "Bene Beraq": "Bnei Brak"
}

unique_locs['order_city'] = unique_locs['order_city'].replace(city_fixes)
df['order_city'] = df['order_city'].replace(city_fixes)

In [32]:
missing_coords = unique_locs[unique_locs['dest_latitude'].isnull() | unique_locs['dest_longitude'].isnull()]
print(f"Number of locations with missing coordinates: {len(missing_coords)}")
# Show failed city-country combinations
print(missing_coords[['order_city', 'order_country']])

Number of locations with missing coordinates: 0
Empty DataFrame
Columns: [order_city, order_country]
Index: []


In [31]:
# Manual coordinate updates
manual_coords = {
    ('Kryvyy Rih', 'Ukraine'): (47.9105, 33.3918),
    ('Wadi as Sir', 'Jordan'): (31.9539, 35.8606),
    ("K'ut'aisi", 'Georgia'): (42.2480, 42.7001),
    ('Cortama', 'Spain'): (37.2242, -4.4662),
    ('Thies Nones', 'Senegal'): (14.7833, -16.9333)
}

for (city, country), (lat, lon) in manual_coords.items():
    mask = (unique_locs['order_city'] == city) & (unique_locs['order_country'] == country)
    unique_locs.loc[mask, 'dest_latitude'] = lat
    unique_locs.loc[mask, 'dest_longitude'] = lon

In [33]:
# Save to CSV
unique_locs.to_csv("unique_locations.csv", index=False)

In [8]:
unique_locs = pd.read_csv("../data/processed/unique_locations.csv")

In [9]:
# Merge unique_locs into df on 'order_city' and 'order_country'
df_lat = pd.merge(
    df,
    unique_locs,
    on=['order_city', 'order_country'],
    how='left'  # Use 'left' join to preserve all rows from df
)

In [10]:
df_lat.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15548 entries, 0 to 15547
Data columns (total 44 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   payment_type              15548 non-null  object 
 1   profit_per_order          15548 non-null  float64
 2   sales_per_customer        15548 non-null  float64
 3   category_id               15548 non-null  int64  
 4   category_name             15548 non-null  object 
 5   customer_city             15548 non-null  object 
 6   customer_country          15548 non-null  object 
 7   customer_id               15548 non-null  float64
 8   customer_segment          15548 non-null  object 
 9   customer_state            15548 non-null  object 
 10  customer_zipcode          15548 non-null  float64
 11  department_id             15548 non-null  int64  
 12  department_name           15548 non-null  object 
 13  latitude                  15548 non-null  float64
 14  longit

In [11]:
# Filter again
retry_df = unique_locs[unique_locs['dest_latitude'].isnull() | unique_locs['dest_longitude'].isnull()]

# Retry with updated city/country
for idx, row in retry_df.iterrows():
    city = row['order_city']
    country = row['order_country']
    lat, lon = get_coordinates(city, country)
    unique_locs.loc[idx, ['dest_latitude', 'dest_longitude']] = lat, lon
    time.sleep(1)

In [12]:
# List of ID-like columns to drop (excluding ZIP code)
cols_to_drop = [
    # Customer-related info unlikely to help with delivery delay:
    'customer_city',
    'customer_state',
    'customer_country',
    'customer_zipcode',
    # 'customer_segment',

    # ID-like features:
    'customer_id',
    'order_id',
    'order_customer_id',
    'order_item_id',
    'order_item_cardprod_id',
    'product_card_id',
    'product_category_id',
    'department_id',
    'category_id',

    # 'product_name', #don't if you are using nlp
    'product_price',  # if duplicate of order_item_product_price
    # 'profit_per_order',  # if duplicate of order_profit_per_order
    # 'sales_per_customer',  # no customer info now
    # 'customer_segment'  # per earlier discussion
    "profit_diff" #created for inspect
]


# Drop those columns
df1 = df_lat.drop(columns=cols_to_drop)

# Check remaining columns
print("Remaining columns:", df1.columns.tolist())

Remaining columns: ['payment_type', 'profit_per_order', 'sales_per_customer', 'category_name', 'customer_segment', 'department_name', 'latitude', 'longitude', 'market', 'order_city', 'order_country', 'order_date', 'order_item_discount', 'order_item_discount_rate', 'order_item_product_price', 'order_item_profit_ratio', 'order_item_quantity', 'sales', 'order_item_total_amount', 'order_profit_per_order', 'order_region', 'order_state', 'order_status', 'product_name', 'shipping_date', 'shipping_mode', 'label', 'dest_latitude', 'dest_longitude']


In [13]:
from geopy.distance import geodesic

# Function to calculate distance between (lat1, lon1) and (lat2, lon2)
def compute_distance(row):
    try:
        store_coords = (row['latitude'], row['longitude'])
        dest_coords = (row['dest_latitude'], row['dest_longitude'])
        if None not in store_coords and None not in dest_coords:
            return geodesic(store_coords, dest_coords).km
        else:
            return None
    except:
        return None

df1['store_to_order_distance_km'] = df1.apply(compute_distance, axis=1)

In [15]:
df1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15548 entries, 0 to 15547
Data columns (total 30 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   payment_type                15548 non-null  object 
 1   profit_per_order            15548 non-null  float64
 2   sales_per_customer          15548 non-null  float64
 3   category_name               15548 non-null  object 
 4   customer_segment            15548 non-null  object 
 5   department_name             15548 non-null  object 
 6   latitude                    15548 non-null  float64
 7   longitude                   15548 non-null  float64
 8   market                      15548 non-null  object 
 9   order_city                  15548 non-null  object 
 10  order_country               15548 non-null  object 
 11  order_date                  15548 non-null  float64
 12  order_item_discount         15548 non-null  float64
 13  order_item_discount_rate    155

In [16]:
import pandas as pd

# Suppose df1 is already loaded
# df1 = pd.read_csv("your_input.csv")

# Step 1: Select required columns
df_out = df1[["order_city", "order_country", "store_to_order_distance_km"]].copy()

# Step 2: Fill missing values with mean
mean_distance = df_out["store_to_order_distance_km"].mean()
df_out["store_to_order_distance_km"].fillna(mean_distance, inplace=True)

# Step 3: Save to CSV
df_out.to_csv("order_city_country_distance.csv", index=False)

print("✅ CSV file created: order_city_country_distance.csv")

✅ CSV file created: order_city_country_distance.csv


C:\Users\VP678WV\AppData\Local\Temp\ipykernel_15616\3336152769.py:11: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_out["store_to_order_distance_km"].fillna(mean_distance, inplace=True)


In [17]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
df1['distance_normalized'] = scaler.fit_transform(df1[['store_to_order_distance_km']])

In [18]:
import joblib

# Save fitted scaler
joblib.dump(scaler, "distance_scaler.pkl")

['distance_scaler.pkl']

In [46]:
print(df1.shape)
print("Remaining columns:", df1.columns.tolist())

(15548, 31)
Remaining columns: ['payment_type', 'profit_per_order', 'sales_per_customer', 'category_name', 'customer_segment', 'department_name', 'latitude', 'longitude', 'market', 'order_city', 'order_country', 'order_date', 'order_item_discount', 'order_item_discount_rate', 'order_item_product_price', 'order_item_profit_ratio', 'order_item_quantity', 'sales', 'order_item_total_amount', 'order_profit_per_order', 'order_region', 'order_state', 'order_status', 'product_name', 'shipping_date', 'shipping_mode', 'label', 'dest_latitude', 'dest_longitude', 'store_to_order_distance_km', 'distance_normalized']


In [47]:
# Save to CSV
df1.to_csv("dropped_new_data.csv", index=False)